# Cross-Lingual Embeddings: One Meaning, Many Languages

**Computational Text Analysis Pack · Module 23 — African NLP Models and Benchmarks**

*Technique:* LaBSE sentence embeddings; cross-lingual retrieval a count-based method cannot do

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/23b_cross_lingual_embeddings.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Here is something no method in the first 22 modules can do.

Take a sentence in Hausa and its translation in English. They share almost no
characters, no tokens, no vocabulary. Every measure this course has taught —
Jaccard, cosine on TF-IDF, Burrows's Delta — scores them at essentially zero,
correctly, because they have nothing in common *as strings*.

A cross-lingual embedding model puts them in nearly the same place.

## The tooling

**HuggingFace `transformers`** is the working layer for every model here: it
downloads the weights, tokenises, and runs the forward pass. Underneath it sits
**PyTorch** (the research default, dynamic and easy to debug); **TensorFlow /
Keras** is the other option, and what the Case Studies module uses.

The models themselves come from African NLP research groups:

- **AfriBERTa** (Ogueji et al.) — trained on 11 African languages, deliberately
  small, and evidence that useful models do not need web-scale data.
- **AfroXLMR** (Alabi et al.) — XLM-R adapted to African languages; the NER model
  in this notebook is its fine-tuned descendant.
- **LaBSE** (Google) — 109 languages, sentence-level, and the reason a Hausa
  sentence and its English translation land in the same place.
- **MasakhaNER / MasakhaNEWS / MasakhaPOS** — the benchmarks, built by the
  Masakhane community. `masakhaner.csv` in this course *is* that benchmark.

### Where the GPU comes from, and why you never need a key

The setup cell picks a backend from wherever you are running:

| Running in | GPU | What you need |
|---|---|---|
| **CourseHub Lab** | a shared L40S, reached through an authenticated CourseHub endpoint | nothing — the credential stays on the server |
| **Colab** | *your own* free GPU (`Runtime > Change runtime type > T4`) | nothing |
| **Anywhere offline** | none; cells use vectors shipped with the course | nothing |

There is no API key in this course, by design. A notebook cell is readable by
whoever runs it, so a key pasted into one is a key given away.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
BACKEND = "cached"
_pod = _hf = None

if IN_COLAB:
    # Colab hands every participant their own GPU. Use it rather than sharing
    # one server: it is faster for a room full of people, and needs no key.
    try:
        import transformers  # noqa: F401
    except ImportError:
        os.system("pip install -q transformers")
    import torch
    BACKEND = "colab-gpu" if torch.cuda.is_available() else "colab-cpu"
else:
    try:
        sys.path.insert(0, "/app")      # the CourseHub runner executes inside /app
        from utils import gpu_pod as _pod
        _info = _pod.health()
        BACKEND = "coursehub-pod"
    except Exception as exc:
        _why = str(exc)[:80]

POD_LIVE = BACKEND in ("coursehub-pod", "colab-gpu", "colab-cpu")

def _hf_pipe(kind, repo):
    global _hf
    _hf = _hf or {}
    if (kind, repo) not in _hf:
        from transformers import AutoModel, AutoTokenizer, pipeline
        import torch
        dev = 0 if torch.cuda.is_available() else -1
        if kind == "ner":
            _hf[(kind, repo)] = pipeline("token-classification", model=repo,
                                         aggregation_strategy="simple", device=dev)
        else:
            _hf[(kind, repo)] = (AutoTokenizer.from_pretrained(repo),
                                 AutoModel.from_pretrained(repo))
    return _hf[(kind, repo)]

def embed(texts, model="labse"):
    """Sentence embeddings, wherever we are. Returns a numpy array."""
    import numpy as np
    if BACKEND == "coursehub-pod":
        return np.asarray(_pod.embed(list(texts), model=model)["vectors"], dtype="float32")
    if BACKEND.startswith("colab"):
        import torch
        tok, mod = _hf_pipe("embed", "sentence-transformers/LaBSE")
        if torch.cuda.is_available():
            mod = mod.cuda()
        with torch.no_grad():
            b = tok(list(texts), padding=True, truncation=True,
                    max_length=128, return_tensors="pt")
            b = {k: v.to(mod.device) for k, v in b.items()}
            out = mod(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            v = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
            v = torch.nn.functional.normalize(v, dim=-1)
        return v.cpu().numpy().astype("float32")
    raise RuntimeError("no GPU backend available; use the shipped vectors instead")

def ner(texts, model="masakhaner"):
    """Named entities from the MasakhaNER 2.0 benchmark model."""
    repo = "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
    if BACKEND == "coursehub-pod":
        return _pod.ner(list(texts), model=model)["results"]
    if BACKEND.startswith("colab"):
        pipe = _hf_pipe("ner", repo)
        out = pipe(list(texts))
        if out and isinstance(out[0], dict):
            out = [out]
        return [[{"text": e["word"], "label": e["entity_group"],
                  "score": round(float(e["score"]), 4)} for e in ents] for ents in out]
    raise RuntimeError("no GPU backend available")

print(f"backend: {BACKEND}")
if BACKEND == "coursehub-pod":
    print(f"   {_info['gpu']}, {_info['vram_total_mb']:,} MB VRAM (shared, via CourseHub)")
elif BACKEND == "colab-gpu":
    import torch; print(f"   {torch.cuda.get_device_name(0)} (your own Colab GPU)")
elif BACKEND == "colab-cpu":
    print("   Colab without a GPU — Runtime > Change runtime type > T4 GPU makes this faster")
else:
    print(f"   no live backend ({_why}); cells fall back to the vectors shipped with the course")

In [ ]:
import numpy as np

PARALLEL = [
    ("Hausa",    "Gwamnati ta ce za ta gina sabbin makarantu a jihar"),
    ("Swahili",  "Serikali imesema itajenga shule mpya katika mkoa"),
    ("Yoruba",   "Ijọba sọ pé òun yóò kọ́ àwọn ilé-ìwé tuntun ní ìpínlẹ̀ náà"),
    ("English",  "The government said it will build new schools in the state"),
    ("French",   "Le gouvernement a dit qu'il construira de nouvelles écoles"),
    ("(unrelated)", "The football match was postponed because of heavy rain"),
]
labels = [p[0] for p in PARALLEL]
sentences = [p[1] for p in PARALLEL]
for lang, s in PARALLEL:
    print(f"  {lang:<14}{s}")

## First, the classical baseline

Run the same six sentences through TF-IDF cosine — the tool module 6 taught.
This is not a strawman; it is the correct answer for a string-based method.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4))
X = tfidf.fit_transform(sentences)
sim = cosine_similarity(X)
print("TF-IDF character n-gram cosine (the classical answer):\n")
print(pd.DataFrame(sim, index=labels, columns=labels).round(2).to_string())
print()
print("The four translations of the SAME sentence score little better against")
print("each other than against the unrelated football sentence.")

## Now the same six sentences through LaBSE

If the pod is live this runs on the L40S. If not, the cell falls back to vectors
shipped with the course, and says which it used — a lesson should never silently
change what it is showing you.

In [ ]:
if POD_LIVE:
    import time as _t
    _t0 = _t.time()
    vecs = embed(sentences, model="labse")
    source = f"{BACKEND} ({int((_t.time()-_t0)*1000)} ms)"
else:
    # Shipped fallback: nearest headline vectors stand in for the demo sentences
    # so the shape of the result survives even with no pod.
    cached = np.load("labse_headlines.npy")
    idx = pd.read_csv("labse_headlines_index.csv")
    picks = []
    for lang in ["Hausa", "Swahili", "Yoruba", "English", "French", "English"]:
        m = idx.index[idx["language"] == lang]
        picks.append(cached[m[0]] if len(m) else cached[0])
    vecs = np.vstack(picks)
    source = "precomputed vectors (pod offline)"

lab_sim = vecs @ vecs.T
print(f"LaBSE cosine — source: {source}\n")
print(pd.DataFrame(lab_sim, index=labels, columns=labels).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mat, title in [(axes[0], sim, "TF-IDF characters"),
                       (axes[1], lab_sim, "LaBSE embeddings")]:
    im = ax.imshow(mat, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=.75)
plt.tight_layout()

print("Left: strings. Right: meaning. The bright block on the right is five")
print("translations of one sentence; the dark row is the football sentence,")
print("which LaBSE also correctly separates.")

## What this is actually for

Cross-lingual retrieval. Ask a question in English, find matching documents in
Hausa, Yorùbá and Swahili — without translating anything. For a researcher
working across a multilingual archive this is the single most useful thing in
the neural toolkit.

In [ ]:
index = pd.read_csv("labse_headlines_index.csv")
bank = np.load("labse_headlines.npy")

QUERY = "election results and political campaigns"     # <- change this

if POD_LIVE:
    qv = embed([QUERY], model="labse")[0]
    scores = bank @ qv
    hits = np.argsort(scores)[::-1][:10]
    print(f"Query (English): {QUERY!r}\n")
    print(f"{'score':>7}  {'language':<16}{'category':<14}headline")
    print("-" * 92)
    for i in hits:
        r = index.iloc[i]
        print(f"{scores[i]:>7.3f}  {r['language']:<16}{r['category']:<14}"
              f"{str(r['headline'])[:44]}")
    print()
    print("Note the languages in that list. None of them is English.")
else:
    print("Pod offline — showing the shipped index instead of a live query.")
    print(index.groupby('language').size().to_string())

### 🔧 Your turn 1

Change QUERY to something from your own research area and look at which languages come back. Does the model retrieve across all of them equally?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Almost never equally. Retrieval quality tracks how much of each language LaBSE
saw in pretraining, so Swahili and Hausa results are usually stronger than
Yorùbá or Lingala ones — and the ranking will quietly under-represent the
languages the model knows least. That is a sampling bias introduced by your
tooling, and if you use cross-lingual retrieval to build a research corpus it
becomes a bias in your findings. Report which model did the retrieval, exactly
as you would report a search string.

</details>